<a href="https://colab.research.google.com/github/Sampavi01/Advanced_Time_Series_Forecasting/blob/time_series/Deep_learning_Models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# --- Core Libraries & Plotting ---
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import io

# --- Helper for Colab File Upload ---
from google.colab import files

# --- Deep Learning Framework (TensorFlow) ---
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, GRU, SimpleRNN, Dense, Dropout, TimeDistributed, Conv1D, MaxPooling1D, Flatten, MultiHeadAttention, LayerNormalization
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

# --- Scikit-Learn Tools ---
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, mean_absolute_percentage_error
from scipy.special import inv_boxcox

# --- Plotting Style ---
plt.style.use('seaborn-v0_8-whitegrid')

In [2]:
# ---  Load Data  ---
from google.colab import files
print("Please upload your 'featured_aep_data.csv' file")
uploaded = files.upload()
print("\n✅ File uploaded successfully!")

Please upload your 'featured_aep_data.csv' file


Saving featured_aep_data.csv to featured_aep_data.csv

✅ File uploaded successfully!


In [3]:
# Next, upload the parameters file.
print("\nNext, please upload your 'model_parameters.joblib' file.")
uploaded_params = files.upload()
print(f"\n✅ Uploaded '{list(uploaded_params.keys())}' successfully!")


Next, please upload your 'model_parameters.joblib' file.


Saving model_parameters.joblib to model_parameters.joblib

✅ Uploaded '['model_parameters.joblib']' successfully!


In [4]:
import io
# --- 2. Load the Uploaded Data and Parameters ---
# Load the DataFrame from the uploaded CSV
df_ml = pd.read_csv(io.BytesIO(uploaded['featured_aep_data.csv']), index_col='Datetime', parse_dates=True)
print("\nFeatured DataFrame successfully loaded.")
print("Shape of loaded data:", df_ml.shape)

params = joblib.load( 'model_parameters.joblib')


Featured DataFrame successfully loaded.
Shape of loaded data: (121247, 24)


In [5]:
# --- Unpack Parameters ---
lambda_boxcox = params['lambda_boxcox']
train_end_idx = params['train_end_idx']
val_end_idx = params['val_end_idx']
TARGET_TRANSFORMED = params['target_col_transformed']
TARGET_ORIGINAL = params['target_col_original']
FEATURES = params['feature_columns']

In [8]:
# ---  Recreate Splits ---
# The original dataset started on '2004-10-01 01:00:00'.
# We can calculate how many rows were dropped by comparing the start date
# of our new DataFrame to the original start date.

original_start_date = pd.to_datetime('2004-10-01 01:00:00')
actual_start_date = df_ml.index.min() # The first timestamp in our loaded data

# The difference in hours is the number of rows that were dropped
time_difference = actual_start_date - original_start_date
rows_dropped = int(time_difference.total_seconds() / 3600)

print(f"Calculated that {rows_dropped} rows were dropped by the feature engineering process.")

# Now, adjust the original split indices by this amount
adjusted_train_end = train_end_idx - rows_dropped
adjusted_val_end = val_end_idx - rows_dropped

# Use the adjusted indices to split the new df_ml DataFrame
train_df = df_ml.iloc[:adjusted_train_end]  # Renamed for clarity, like in your original code
val_df = df_ml.iloc[adjusted_train_end:adjusted_val_end]
test_df = df_ml.iloc[adjusted_val_end:]

Calculated that 49 rows were dropped by the feature engineering process.


In [9]:
# --- Scale the Data ---
# Neural networks require input features to be scaled, typically between 0 and 1.
# IMPORTANT: We fit the scaler ONLY on the training data to prevent data leakage.
scaler = MinMaxScaler()
# We scale both features and the target together for easier sequence creation.
train_scaled = scaler.fit_transform(train_df[FEATURES + [TARGET_TRANSFORMED]])
val_scaled = scaler.transform(val_df[FEATURES + [TARGET_TRANSFORMED]])
test_scaled = scaler.transform(test_df[FEATURES + [TARGET_TRANSFORMED]])

print("\n✅ Data is loaded, split, and scaled. Ready for sequence creation.")


✅ Data is loaded, split, and scaled. Ready for sequence creation.


### ** Data Preparation for Sequence Models**

This is the most important new concept for deep learning. RNNs (LSTMs, GRUs) see a **sequence** of past data to predict the future. We must transform our 2D data (rows, features) into 3D data: `(samples, timesteps, features)`.

-   **`timesteps`**: How many hours of past data the model looks at (e.g., the last 7 days). This is our "lookback window."

In [10]:
def create_sequences(data, sequence_length, target_index):
    """Creates sequences of data for time series forecasting."""
    X, y = [], []
    for i in range(len(data) - sequence_length):
        # The input sequence is the window of past data
        X.append(data[i:(i + sequence_length), :])
        # The target is the value of the target variable at the end of the window
        y.append(data[i + sequence_length, target_index])
    return np.array(X), np.array(y)

# --- Define Hyperparameters ---
SEQUENCE_LENGTH = 24 * 7 # Look back at the last 7 days of hourly data (168 hours)
TARGET_INDEX = len(FEATURES) # The target is the last column in our scaled data array

# --- Create the sequences for training, validation, and testing ---
X_train_seq, y_train_seq = create_sequences(train_scaled, SEQUENCE_LENGTH, TARGET_INDEX)
X_val_seq, y_val_seq = create_sequences(val_scaled, SEQUENCE_LENGTH, TARGET_INDEX)
X_test_seq, y_test_seq = create_sequences(test_scaled, SEQUENCE_LENGTH, TARGET_INDEX)

print("\n✅ Data successfully transformed into sequences.")
print(f"X_train_seq shape: {X_train_seq.shape}") # Should be (samples, 168, 22)
print(f"y_train_seq shape: {y_train_seq.shape}")
print(f"X_val_seq shape: {X_val_seq.shape}")


✅ Data successfully transformed into sequences.
X_train_seq shape: (84674, 168, 22)
y_train_seq shape: (84674,)
X_val_seq shape: (18023, 168, 22)


### ** Baseline Models: LSTM vs. GRU**

We will build our first deep learning models. LSTMs and GRUs are types of Recurrent Neural Networks (RNNs) that are excellent at learning from sequential data. We'll build them with identical architectures for a fair comparison.


In [11]:
from tensorflow.keras.layers import  Input
# --- Define Callbacks ---
early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Get the number of features
num_features = X_train_seq.shape[2]  # feature dimension

# --- Build LSTM Model ---
lstm_model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, num_features)),  # only timesteps & features
    LSTM(64),
    Dropout(0.2),
    Dense(1)
])
lstm_model.compile(optimizer='adam', loss='mean_squared_error')
lstm_model.summary()

# --- Build GRU Model ---
gru_model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, num_features)),
    GRU(64),
    Dropout(0.2),
    Dense(1)
])
gru_model.compile(optimizer='adam', loss='mean_squared_error')
gru_model.summary()

# --- Train LSTM Model ---
history_lstm = lstm_model.fit(
    X_train_seq, y_train_seq,
    epochs=50, batch_size=64,
    validation_data=(X_val_seq, y_val_seq),
    callbacks=[early_stopping],
    verbose=1
)

# --- Train GRU Model ---
history_gru = gru_model.fit(
    X_train_seq, y_train_seq,
    epochs=50, batch_size=64,
    validation_data=(X_val_seq, y_val_seq),
    callbacks=[early_stopping],
    verbose=1
)

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        22,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 22,337 (87.25 KB)

 Trainable params: 22,337 (87.25 KB)

 Non-trainable params: 0 (0.00 B)

...

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ gru (GRU)                       │ (None, 64)             │        16,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 16,961 (66.25 KB)

 Trainable params: 16,961 (66.25 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 17s 10ms/step - loss: 0.0191 - val_loss: 0.0011
Epoch 2/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 12s 9ms/step - loss: 0.0021 - val_loss: 5.8056e-04
Epoch 3/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - loss: 0.0013 - val_loss: 6.6825e-04
Epoch 4/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 7.8747e-04 - val_loss: 3.9499e-04
Epoch 5/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 5.3332e-04 - val_loss: 2.4027e-04
Epoch 6/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 4.0208e-04 - val_loss: 2.3876e-04
Epoch 7/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 21s 9ms/step - loss: 3.5029e-04 - val_loss: 1.5794e-04
Epoch 8/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 3.3694e-04 - val_loss: 5.2299e-04
Epoch 9/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 3.1258e-04 - val_loss: 1.3735e-04
Epoch 10/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 9ms/step - loss: 3.0815e-04 - val_loss: 1.8636e-04
Epoch 11/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 2

### ** Evaluation Function and Baseline Results**

We need a robust function to evaluate our models. This function must handle predicting, inverse scaling, and inverse Box-Cox transforming to get the final error in real-world Megawatts.

In [12]:
def evaluate_model(model, X_test_seq, y_test_orig, scaler, lambda_boxcox, sequence_length):
    """Makes predictions, inverts scaling/transformations, and calculates RMSE."""
    # 1. Predict on the scaled test data
    preds_scaled = model.predict(X_test_seq)

    # 2. Inverse scale the predictions
    # We create a dummy array of the same shape as the original data (features + target)
    dummy_array = np.zeros((len(preds_scaled), scaler.n_features_in_))
    # We place our predictions into the target column
    dummy_array[:, -1] = preds_scaled.ravel()
    # Now we can inverse transform
    preds_boxcox = scaler.inverse_transform(dummy_array)[:, -1]

    # 3. Inverse Box-Cox transform
    preds_orig = inv_boxcox(preds_boxcox, lambda_boxcox)

    # 4. Align with original test set and calculate RMSE
    # The first 'sequence_length' values of y_test_orig have no prediction
    true_values = y_test_orig.iloc[sequence_length:]
    rmse = np.sqrt(mean_squared_error(true_values, preds_orig))
    return rmse

# --- Store results for comparison ---
model_results = {}


In [13]:
# Evaluate LSTM
lstm_rmse = evaluate_model(lstm_model, X_test_seq, test_df[TARGET_ORIGINAL], scaler, lambda_boxcox, SEQUENCE_LENGTH)
model_results['LSTM (Baseline)'] = lstm_rmse
print(f"LSTM Model Test RMSE: {lstm_rmse:.2f} MW")

# Evaluate GRU
gru_rmse = evaluate_model(gru_model, X_test_seq, test_df[TARGET_ORIGINAL], scaler, lambda_boxcox, SEQUENCE_LENGTH)
model_results['GRU (Baseline)'] = gru_rmse
print(f"GRU Model Test RMSE: {gru_rmse:.2f} MW")

564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
LSTM Model Test RMSE: 149.19 MW
564/564 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step
GRU Model Test RMSE: 157.29 MW


### ** Advanced Model 1: Hybrid CNN-LSTM**

Here, we use a 1D Convolutional Neural Network (CNN) to act as a feature extractor. The CNN will scan the 168-hour input sequence to identify important local patterns (like daily peaks) before the LSTM layer processes the sequence of these extracted patterns.

In [15]:
# --- Build the CNN-LSTM Model ---
cnn_lstm_model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, num_features)),  # only timesteps & features
    Conv1D(filters=64, kernel_size=3, activation='relu'),
    MaxPooling1D(pool_size=2),
    LSTM(64, activation='relu'),
    Dropout(0.2),
    Dense(1)
])

cnn_lstm_model.compile(optimizer='adam', loss='mean_squared_error')
cnn_lstm_model.summary()
# --- Train the CNN-LSTM Model ---
print("\n--- Training CNN-LSTM Model ---")
history_cnn_lstm = cnn_lstm_model.fit(X_train_seq, y_train_seq,
                                      epochs=50, batch_size=64,
                                      validation_data=(X_val_seq, y_val_seq),
                                      callbacks=[early_stopping], verbose=1)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_1 (Conv1D)               │ (None, 166, 64)        │         4,288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 83, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_2 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 37,377 (146.00 KB)

 Trainable params: 37,377 (146.00 KB)

 Non-trainable params: 0 (0.00 B)


--- Training CNN-LSTM Model ---
Epoch 1/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 24s 13ms/step - loss: 0.0135 - val_loss: 9.3521e-04
Epoch 2/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - loss: 0.0022 - val_loss: 5.8491e-04
Epoch 3/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - loss: 0.0013 - val_loss: 4.4707e-04
Epoch 4/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - loss: 8.8685e-04 - val_loss: 0.0032
Epoch 5/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 21s 10ms/step - loss: 0.0011 - val_loss: 0.0014
Epoch 6/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - loss: 7.5782e-04 - val_loss: 3.6910e-04
Epoch 7/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - loss: 6.7381e-04 - val_loss: 2.1028e-04
Epoch 8/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 13s 10ms/step - loss: 6.4092e-04 - val_loss: 3.7592e-04
Epoch 9/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 14s 10ms/step - loss: 6.3218e-04 - val_loss: 9.8491e-04
Epoch 10/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 20s 10ms/step - loss: 6.2785e-04 - val_loss: 2.7499e-04
Epoch 11/5

In [16]:
# --- Evaluate the CNN-LSTM Model ---
cnn_lstm_rmse = evaluate_model(cnn_lstm_model, X_test_seq, test_df[TARGET_ORIGINAL], scaler, lambda_boxcox, SEQUENCE_LENGTH)
model_results['CNN-LSTM'] = cnn_lstm_rmse
print(f"CNN-LSTM Model Test RMSE: {cnn_lstm_rmse:.2f} MW")

564/564 ━━━━━━━━━━━━━━━━━━━━ 3s 5ms/step
CNN-LSTM Model Test RMSE: 172.13 MW


### ** Advanced Model 2: Transformer**

This is the state-of-the-art architecture. Unlike RNNs that process data sequentially, the Transformer uses a **self-attention** mechanism to weigh the importance of all input timesteps simultaneously, allowing it to capture complex long-range dependencies.


In [22]:
from keras.models import Model
from tensorflow.keras.layers import GlobalAveragePooling1D

def transformer_encoder(inputs, head_size, num_heads, ff_dim, dropout=0):
    # Attention and Normalization
    x = MultiHeadAttention(key_dim=head_size, num_heads=num_heads, dropout=dropout)(inputs, inputs)
    x = Dropout(dropout)(x)
    x = LayerNormalization(epsilon=1e-6)(x + inputs) # Add & Norm

    # Feed Forward Part
    ff_x = Dense(ff_dim, activation="relu")(x)
    ff_x = Dropout(dropout)(ff_x)
    ff_x = Dense(inputs.shape[-1])(ff_x)
    return LayerNormalization(epsilon=1e-6)(x + ff_x) # Add & Norm

def build_transformer_model(input_shape, head_size, num_heads, ff_dim, num_transformer_blocks, mlp_units, dropout=0, mlp_dropout=0):
    inputs = Input(shape=input_shape)
    x = inputs
    for _ in range(num_transformer_blocks):
        x = transformer_encoder(x, head_size, num_heads, ff_dim, dropout)

    x = GlobalAveragePooling1D()(x)
    for dim in mlp_units:
        x = Dense(dim, activation="relu")(x)
        x = Dropout(mlp_dropout)(x)
    outputs = Dense(1)(x)
    return Model(inputs, outputs)

In [23]:
# --- Build and Train the Transformer ---
input_shape = (X_train_seq.shape[1], X_train_seq.shape[2])  # (seq_len, num_features)

transformer_model = build_transformer_model(
    input_shape, head_size=64, num_heads=4, ff_dim=64,
    num_transformer_blocks=4, mlp_units=[64, 32], mlp_dropout=0.4, dropout=0.25
)

transformer_model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),
    loss="mean_squared_error"
)

print("--- Transformer Model Summary ---")
transformer_model.summary()

history_transformer = transformer_model.fit(
    X_train_seq, y_train_seq,
    epochs=50, batch_size=64,
    validation_data=(X_val_seq, y_val_seq),
    callbacks=[early_stopping], verbose=1
)


--- Transformer Model Summary ---


Model: "functional_3"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6       │ (None, 168, 22)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 168, 22)   │     23,318 │ input_layer_6[0]… │
│ (MultiHeadAttentio… │                   │            │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_17          │ (None, 168, 22)   │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_8 (Add)         │ (None, 168, 22)   │          0 │ dropout_17[0][0], │
│                     │                   │            │ input_layer_6[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 168, 22)   │         44 │ add_8[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 168, 64)   │      1,472 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_18          │ (None, 168, 64)   │          0 │ dense_12[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_13 (Dense)    │ (None, 168, 22)   │      1,430 │ dropout_18[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_9 (Add)         │ (None, 168, 22)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_13[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 168, 22)   │         44 │ add_9[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 168, 22)   │     23,318 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_20          │ (None, 168, 22)   │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_10 (Add)        │ (None, 168, 22)   │          0 │ dropout_20[0][0], │
│                     │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 168, 22)   │         44 │ add_10[0][0]      │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_14 (Dense)    │ (None, 168, 64)   │      1,472 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_21          │ (None, 168, 64)   │          0 │ dense_14[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_15 (Dense)    │ (None, 168, 22)   │      1,430 │ dropout_21[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_11 (Add)        │ (None, 168, 22)   │          0 │ layer_normalizat… │
│                     │                   │            │ dense_15[0][0]  

 Total params: 108,817 (425.07 KB)

 Trainable params: 108,817 (425.07 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 76s 40ms/step - loss: 0.0922 - val_loss: 0.0246
Epoch 2/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 58s 32ms/step - loss: 0.0332 - val_loss: 0.0230
Epoch 3/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 80s 30ms/step - loss: 0.0271 - val_loss: 0.0207
Epoch 4/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 41s 30ms/step - loss: 0.0247 - val_loss: 0.0235
Epoch 5/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 40s 30ms/step - loss: 0.0238 - val_loss: 0.0219
Epoch 6/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 43s 32ms/step - loss: 0.0224 - val_loss: 0.0202
Epoch 7/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 82s 32ms/step - loss: 0.0216 - val_loss: 0.0202
Epoch 8/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 82s 32ms/step - loss: 0.0212 - val_loss: 0.0209
Epoch 9/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 82s 32ms/step - loss: 0.0206 - val_loss: 0.0217
Epoch 10/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 79s 30ms/step - loss: 0.0202 - val_loss: 0.0212
Epoch 11/50
1324/1324 ━━━━━━━━━━━━━━━━━━━━ 42s 32ms/step - loss: 0.0200 - val_loss: 0.0210
Epoch 12

In [24]:
# --- Evaluate the Transformer Model ---
transformer_rmse = evaluate_model(transformer_model, X_test_seq, test_df[TARGET_ORIGINAL], scaler, lambda_boxcox, SEQUENCE_LENGTH)
model_results['Transformer'] = transformer_rmse
print(f"Transformer Model Test RMSE: {transformer_rmse:.2f} MW")

564/564 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step
Transformer Model Test RMSE: 779.15 MW


## ** Final Comparison and Conclusion**

Finally, let's compare the performance of all our deep learning models against each other


In [26]:
# --- Final Comparison ---
results_df = pd.DataFrame.from_dict(model_results, orient='index', columns=['Test_RMSE_MW'])
print("\n\n--- Final Deep Learning Model Comparison (Sorted by Test RMSE) ---")
print(results_df.sort_values('Test_RMSE_MW'))




--- Final Deep Learning Model Comparison (Sorted by Test RMSE) ---
                 Test_RMSE_MW
LSTM (Baseline)    149.189942
GRU (Baseline)     157.289415
CNN-LSTM           172.130767
Transformer        779.150744
